In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(
    "Abastecimento.csv",
    sep=";",        # separador correto
    decimal=".",    # decimal brasileiro
    encoding="latin1"  # comum em arquivos exportados do Excel
)

In [3]:
df = df[["ANO","DATA", "VEICULO", "TIPO","GASOLINA", "ALCOOL", "DIESEL", "GAS", "KM", "VALOR", "KM RODADA", "MEDIA L/KM", "MOTORISTA", "DESCONTOS"]]

In [4]:
try:
    from readline import redisplay  # Linux/macOS
except ModuleNotFoundError:
    redisplay = None  # Windows: ignora ou trate de outra forma
import pandas as pd
import numpy as np
df.columns = df.columns.str.strip()
df_model = df[["VEICULO", "KM RODADA", "DIESEL"]].copy()
df_model = df_model.dropna(how="all")
df_model["VEICULO"] = df_model["VEICULO"].astype(str).str.strip()
# KM RODADA: já é numérico → só garante conversão
df_model["KM RODADA"] = pd.to_numeric(df_model["KM RODADA"], errors="coerce")
# DIESEL: vem como string BR → troca vírgula por ponto (NÃO remove ".")
s = df_model["DIESEL"].astype(str).str.strip()
s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "NaN": np.nan})
s = s.str.replace(r"\s+", "", regex=True)
s = s.str.replace(",", ".", regex=False)
df_model["DIESEL"] = pd.to_numeric(s, errors="coerce")
# Remover inválidos
df_model = df_model.dropna(subset=["VEICULO", "KM RODADA", "DIESEL"])
df_model = df_model[(df_model["KM RODADA"] > 0) & (df_model["DIESEL"] > 0)]
print("Após limpeza:", df_model.shape)
if redisplay:
    redisplay()
print(df_model.head())

Após limpeza: (5461, 3)
     VEICULO  KM RODADA  DIESEL
12  PLL 6A56      654.0  29.674
19  PLI 3471      571.0  55.630
21  PLJ 2A71      393.0  41.511
25  PLI 3471      365.0  32.241
29  PLJ 2A71      394.0  37.190


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [6]:
X = df_model[["VEICULO", "KM RODADA"]]
y = df_model["DIESEL"]

cat_features = ["VEICULO"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    random_seed=42,
    verbose=False
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

In [9]:
preds = model.predict(X_test)

print("MAE (erro médio em litros):", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))

MAE (erro médio em litros): 4.035694177515348
R²: 0.8946966389349341


In [10]:
def recomendar_veiculos(km_rota, top_n=5):
    veiculos = df_model[["VEICULO"]].drop_duplicates().reset_index(drop=True)

    veiculos["KM RODADA"] = km_rota
    veiculos["LITROS_PREVISTOS"] = model.predict(veiculos)

    ranking = veiculos.sort_values("LITROS_PREVISTOS").head(top_n)
    return ranking

In [11]:
recomendar_veiculos(km_rota=417)

,VEICULO,KM RODADA,LITROS_PREVISTOS
3,PLQ 6I57,417,37.413501
7,PLR 4F49,417,37.413501
8,PLR 8E03,417,39.763718
2,PLJ 2A71,417,39.855066
1,PLI 3471,417,39.940792
